# arXiv embeddings at scale

This notebook consumes the sample and text-vector cache made by [03a: Prepare the arXiv dataset](03a_prepare_arxiv_dataset.ipynb). It fits cosine-distance UMAP and two cosine-distance IMSTE starts only when the matching coordinates are not already cached. The saved layouts can be explored in notebook 04.

## 1. Load the prepared dataset

Run `03a_prepare_arxiv_dataset.ipynb` first if there is no cache or if you want a larger sample. Here, increasing or changing model settings does not download or re-embed text; this notebook reads the one prepared cache.

In [ ]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
DATA_DIR = PROJECT_ROOT / "data" / "arxiv"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
from arxiv_dataset import load_arxiv_dataset
papers, X, data_manifest = load_arxiv_dataset(DATA_DIR)
print(f"Loaded {len(papers):,} papers × {X.shape[1]:,} features")
print(f"Vector matrix in memory: {X.nbytes / 1024**3:.2f} GiB")
print("Dataset cache ID:", data_manifest["dataset_id"])


## 2. Prepare category labels

In [ ]:
primary_category = papers["categories"].fillna("unknown").str.split().str[0]
archive = primary_category.str.split(".").str[0]
domain_names = {
    "cs": "computer science",
    "math": "mathematics",
    "eess": "engineering",
    "stat": "statistics",
    "q-bio": "biology",
    "q-fin": "finance",
    "econ": "economics",
}
domain = archive.map(domain_names).fillna("physics")
domain_order = domain.value_counts().index.tolist()
labels = pd.Categorical(domain, categories=domain_order)
display(pd.DataFrame({"primary_category": primary_category, "domain": domain}).groupby("domain", observed=True).size().sort_values(ascending=False).rename("papers"))

## 3. Reuse or fit UMAP and IMSTE coordinates

Projection settings and the prepared data ID determine cache reuse. When settings change, the cache is replaced with one matching set of coordinates.

In [ ]:
import hashlib
import time
import umap

RANDOM_STATE = int(data_manifest["random_state"])
N_NEIGHBORS = 15
UMAP_EPOCHS = 200
UMAP_REPULSION_STRENGTH = 8
IMSTE_N_MSTS = 4
IMSTE_N_EPOCHS = 400
IMSTE_NEGATIVE_RATIO = 1
IMSTE_LAMBDA_REP = 0.5
IMSTE_MST_METHOD = "famst"
IMSTE_MST_NEIGHBORS = 5
IMSTE_DEVICE = "mps"
MULTILEVEL_REDUCTION_RATIO = 4
MULTILEVEL_BASE_SIZE = 1024
MULTILEVEL_EPOCHS = 10

projection_settings = {
    "data_id": data_manifest["dataset_id"],
    "umap": [N_NEIGHBORS, UMAP_EPOCHS, UMAP_REPULSION_STRENGTH, 0.1, "cosine", "rinnd", RANDOM_STATE],
    "imste": [IMSTE_N_MSTS, IMSTE_N_EPOCHS, IMSTE_NEGATIVE_RATIO, IMSTE_LAMBDA_REP,
              IMSTE_MST_METHOD, IMSTE_MST_NEIGHBORS, IMSTE_DEVICE, RANDOM_STATE],
    "multilevel": [MULTILEVEL_REDUCTION_RATIO, MULTILEVEL_BASE_SIZE, MULTILEVEL_EPOCHS],
}
projection_id = hashlib.sha256(json.dumps(projection_settings, sort_keys=True).encode()).hexdigest()[:16]
PROJECTION_CACHE = CACHE_DIR / "arxiv_embeddings.npz"
if PROJECTION_CACHE.is_file():
    with np.load(PROJECTION_CACHE, allow_pickle=False) as saved:
        cache_hit = str(saved["projection_id"].item()) == projection_id
else:
    cache_hit = False

if cache_hit:
    with np.load(PROJECTION_CACHE, allow_pickle=False) as saved:
        Z_umap = saved["umap"]
        Z_imste = saved["imste_umap_start"]
        Z_imste_multilevel = saved["imste_multilevel"]
        umap_seconds = float(saved["umap_seconds"])
        imste_seconds = float(saved["imste_seconds"])
        multilevel_seconds = float(saved["multilevel_seconds"])
    print("Reused cached UMAP and IMSTE coordinates:", PROJECTION_CACHE)
else:
    umap_started = time.perf_counter()
    Z_umap = umap.UMAP(
        n_components=2,
        n_neighbors=N_NEIGHBORS,
        min_dist=0.1,
        repulsion_strength=UMAP_REPULSION_STRENGTH,
        metric="cosine",
        compatibility_layout=False,
        nearest_neighbor_backend="rinnd",
        init="recursive",
        random_state=RANDOM_STATE,
        n_epochs=UMAP_EPOCHS,
        verbose=True,
    ).fit_transform(X)
    umap_seconds = time.perf_counter() - umap_started
    print(f"UMAP fit: {umap_seconds / 60:.1f} min")

    from imste._estimator import IteratedMinimumSpanningTreeEmbedder
    settings = dict(
        n_components=2, n_msts=IMSTE_N_MSTS, n_epochs=IMSTE_N_EPOCHS,
        negative_ratio=IMSTE_NEGATIVE_RATIO, lambda_rep=IMSTE_LAMBDA_REP,
        mst_method=IMSTE_MST_METHOD, mst_neighbors=IMSTE_MST_NEIGHBORS,
        random_state=RANDOM_STATE, device=IMSTE_DEVICE,
    )
    started = time.perf_counter()
    Z_imste = IteratedMinimumSpanningTreeEmbedder(**settings, init_coordinates=Z_umap).fit_transform(X)
    imste_seconds = time.perf_counter() - started
    print(f"UMAP-initialized IMSTE fit: {imste_seconds / 60:.1f} min")

    started = time.perf_counter()
    Z_imste_multilevel = IteratedMinimumSpanningTreeEmbedder(
        **settings, multilevel_init=True,
        multilevel_reduction_ratio=MULTILEVEL_REDUCTION_RATIO,
        multilevel_base_size=MULTILEVEL_BASE_SIZE,
        multilevel_epochs=MULTILEVEL_EPOCHS,
    ).fit_transform(X)
    multilevel_seconds = time.perf_counter() - started
    print(f"Multilevel-initialized IMSTE fit: {multilevel_seconds / 60:.1f} min")

    temp_path = PROJECTION_CACHE.with_suffix(".tmp.npz")
    np.savez_compressed(
        temp_path, projection_id=np.asarray(projection_id),
        umap=np.asarray(Z_umap, dtype=np.float32),
        imste_umap_start=np.asarray(Z_imste, dtype=np.float32),
        imste_multilevel=np.asarray(Z_imste_multilevel, dtype=np.float32),
        labels=np.asarray(labels.astype(str)),
        umap_seconds=np.asarray(umap_seconds), imste_seconds=np.asarray(imste_seconds),
        multilevel_seconds=np.asarray(multilevel_seconds),
        dataset_id=np.asarray(data_manifest["dataset_id"]),
        settings_json=np.asarray(json.dumps(projection_settings, sort_keys=True)),
    )
    temp_path.replace(PROJECTION_CACHE)
    print("Saved reusable projection cache:", PROJECTION_CACHE)


## 4. Compare fit runtimes

In [ ]:
runtime_summary = pd.DataFrame({
    "method": ["UMAP", "IMSTE · UMAP start", "IMSTE · multilevel start"],
    "seconds": [umap_seconds, imste_seconds, multilevel_seconds],
})
runtime_summary["runtime"] = runtime_summary["seconds"].map(
    lambda value: f"{value / 60:.1f} min" if value >= 60 else f"{value:.1f} s"
)
display(runtime_summary)
